# FVEG22 Vegetation Comprehensive Analysis
## California Wildfire Prediction Model — Phase 2B

**Dataset**: FRAP FVEG22 — California Wildlife Habitat Relationships (CWHR) Vegetation Classification  
**Author**: Tam Air Club / Olivier  
**Date**: March 2026  
**Resolution**: 30m × 30m statewide raster  
**Classification**: 65 WHRTYPE → 13 WHR13 → 8 LIFEFORM categories

### Table of Contents

| Part | Title | Key Outputs |
|------|-------|-------------|
| 1 | Introduction & Context | CWHR system overview, dataset comparison |
| 2 | Data Loading & Quality | Metadata, valid/NoData coverage, RAT resolution |
| 3 | Value Distribution | Raw pixel values, WHRTYPE breakdown |
| 4 | Lifeform & WHR13 Classification | 8 lifeform composition, 13-class WHR13 bars |
| 5 | Spatial Analysis | Statewide vegetation map, regional composition |
| 6 | Fire-Relevant Vegetation | Fire-prone types, fuel classification map |
| 7 | Vegetation Structure | Tree size class and canopy density analysis |
| 8 | Fire Overlay | Burned vs unburned vegetation, fire-vegetation crosstab |
| 9 | 800m Grid & ML Readiness | Resolution comparison, ML feature recommendations |

---
## Part 1: Introduction & Context

### Why Vegetation Data Matters for Wildfire Prediction

Vegetation type is one of the strongest predictors of wildfire behavior:
- **Fuel type** determines fire intensity and spread rate
- **Chaparral and dry grassland** are California's most fire-prone vegetation
- **Conifer forests** support crown fires in drought conditions
- **Fuel structure** (size class, density) affects flame length and spotting distance

### CWHR Classification System

The California Wildlife Habitat Relationships (CWHR) system classifies vegetation into:
- **65 WHRTYPE** codes — individual vegetation communities (e.g., SMC = Sierran Mixed Conifer)
- **13 WHR13** aggregated classes — broader land cover groups
- **8 LIFEFORM** categories — top-level grouping (Conifer, Hardwood, Shrub, etc.)

Each pixel also encodes:
- **WHRSIZE** (1–6, X) — tree size class / structural stage
- **WHRDENSITY** (S, P, M, D, X) — canopy density class

### FVEG22 Dataset vs Other Vegetation Sources

| Feature | FVEG22 (this analysis) | LANDFIRE CC (Phase 2B-CC) |
|---------|----------------------|--------------------------|
| Question answered | What vegetation types? | How dense is the canopy? |
| Classification | CWHR types (65 classes) | Canopy cover % (10 classes) |
| Resolution | 30m | 30m |
| Extent | California only | CONUS |
| Source | CAL FIRE FRAP + CDFW VegCAMP | USGS/USFS LANDFIRE |
| Temporal | ~1990–2022 compilation | 2024 snapshot |

---
## Part 2: Data Loading & Quality

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────

# Standard library
import sys, warnings
warnings.filterwarnings('ignore')

# Scientific stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns

# Geospatial
import rasterio
from rasterio.features import geometry_mask
import geopandas as gpd

# Project utilities
sys.path.insert(0, '..')
from src.utils.constants import (
    FVEG22_PATH, FVEG22_NODATA, FVEG22_NATIVE_RES_M,
    FVEG22_VIZ_DOWNSAMPLE, FVEG22_RAT_CSV_PATH,
    FIRE_PERIMETERS_PATH, MTBS_FOD_PATH,
    CRS_ALBERS, CRS_WGS84, FIGURES_DIR, FIGURE_DPI,
    HIGH_QUALITY_CUTOFF, CA_BOUNDARY_PATH,
)
from src.utils.vegetation_domain_mappings import (
    WHRTYPE_MAP, LIFEFORM_MAP, LIFEFORM_ORDER, WHR13_MAP, WHR13_ORDER,
    WHRTYPE_TO_LIFEFORM, WHRTYPE_TO_WHR13,
    WHRSIZE_MAP, WHRSIZE_ORDER, WHRDENSITY_MAP, WHRDENSITY_ORDER,
    FIRE_PRONE_WHRTYPE, FIRE_RISK_ORDER,
    get_whrtype_name, get_lifeform, get_whr13_class, get_fire_risk_level,
    parse_whrall, classify_fire_fuel_type,
)
from src.utils.vegetation_data_loader import (
    load_vegetation_metadata, load_vegetation_downsampled,
    get_unique_values_and_counts, get_value_to_attributes,
    merge_values_with_rat, extract_vegetation_at_points,
    get_vegetation_data_quality_summary,
)
from src.utils.plotting import (
    setup_plotting_style, save_figure,
    get_lifeform_colors, get_whr13_colors, get_fire_risk_colors,
)

# Setup
setup_plotting_style()
FIG_DIR = FIGURES_DIR / "vegetation"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Imports complete.")

In [ ]:
# ── Load metadata ────────────────────────────────────────────────────────

meta = load_vegetation_metadata()
print("FVEG22 Raster Metadata:")
for k, v in meta.items():
    if k != 'transform':
        print(f"  {k}: {v}")

total_pixels = meta['height'] * meta['width']
print(f"\nTotal pixels: {total_pixels:,}")
print(f"Estimated valid: ~{total_pixels * 0.3056:,.0f} (30.56%)")

In [ ]:
# ── Load downsampled raster (40x → ~1200m) ──────────────────────────────

veg_ds, veg_transform, veg_meta = load_vegetation_downsampled()
print(f"Downsampled shape: {veg_meta['downsampled_shape']}")
print(f"Effective resolution: {veg_meta['effective_resolution_m']:.0f}m")

# Quality summary
quality = get_vegetation_data_quality_summary(veg_ds)
print("\nData Quality (downsampled):")
for k, v in quality.items():
    print(f"  {k}: {v}")

In [ ]:
# ── Load RAT (Raster Attribute Table) ────────────────────────────────────

local_rat, ref_rat, rat_available = get_value_to_attributes()

if rat_available:
    print(f"\nLocal RAT columns: {list(local_rat.columns)}")
    print(f"Sample mappings:")
    print(local_rat.head(5))

if ref_rat is not None:
    # Build WHRTYPE reference table from ImageServer RAT
    whrtype_ref = ref_rat.groupby('WHRTYPE').agg({
        'WHRNAME': 'first',
        'LIFE_FORM': 'first',
        'WHR13NAME': 'first',
        'COUNT': 'sum',
    }).reset_index().sort_values('COUNT', ascending=False)
    print(f"\nWHRTYPE reference: {len(whrtype_ref)} types")

In [ ]:
# ── Figure 01: Data Quality Overview ─────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel 1: Valid vs NoData coverage
valid_mask = ~np.isnan(veg_ds)
ax = axes[0]
ax.imshow(valid_mask.astype(int), cmap='RdYlGn', aspect='equal',
          extent=[meta['bounds'].left, meta['bounds'].right,
                  meta['bounds'].bottom, meta['bounds'].top])
ax.set_title('Valid Data Coverage', fontsize=13, fontweight='bold')
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='green', label='Valid'),
                   Patch(facecolor='red', label='NoData')]
ax.legend(handles=legend_elements, loc='lower left', fontsize=9)

# Panel 2: Value histogram (valid pixels only)
ax = axes[1]
valid_vals = veg_ds[valid_mask].ravel()
ax.hist(valid_vals, bins=100, color='#2980B9', edgecolor='none', alpha=0.8)
ax.set_title('Pixel Value Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Pixel Value')
ax.set_ylabel('Frequency')
ax.set_yscale('log')
ax.annotate(f'Unique values: {len(np.unique(valid_vals)):,}\n'
            f'Valid pixels: {int(np.sum(valid_mask)):,}',
            xy=(0.97, 0.97), xycoords='axes fraction',
            ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

# Panel 3: Quality metrics table
ax = axes[2]
ax.axis('off')
table_data = [[k, v] for k, v in quality.items()]
table = ax.table(cellText=table_data, colLabels=['Metric', 'Value'],
                 loc='center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2980B9')
        cell.set_text_props(color='white', fontweight='bold')
    else:
        cell.set_facecolor('#F8F9FA' if row % 2 == 0 else 'white')
ax.set_title('Quality Summary', fontsize=13, fontweight='bold', pad=20)

fig.suptitle('Figure 01: FVEG22 Data Quality Overview', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'vegetation/01_data_quality_overview')
plt.show()

---
## Part 3: Value Distribution

In [ ]:
# ── Scan unique values (chunked for memory efficiency) ───────────────────

# Use downsampled data for quick analysis; full scan takes ~3s
ds_valid = veg_ds[~np.isnan(veg_ds)].ravel().astype(int)
unique_ds, counts_ds = np.unique(ds_valid, return_counts=True)
val_counts_ds = pd.DataFrame({'VALUE': unique_ds, 'COUNT': counts_ds})
val_counts_ds = val_counts_ds.sort_values('COUNT', ascending=False).reset_index(drop=True)

print(f"Unique values (downsampled): {len(val_counts_ds):,}")
print(f"Top 10 values by pixel count:")
print(val_counts_ds.head(10).to_string(index=False))

In [ ]:
# ── Figure 02: Raw Value Distribution ────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Value histogram
ax = axes[0]
ax.hist(ds_valid, bins=200, color='#2980B9', edgecolor='none', alpha=0.8)
ax.set_title('Value Histogram', fontsize=12, fontweight='bold')
ax.set_xlabel('Pixel Value')
ax.set_ylabel('Frequency (log)')
ax.set_yscale('log')

# Panel 2: CDF
ax = axes[1]
sorted_vals = np.sort(ds_valid)
cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
# Subsample for plotting
step = max(1, len(sorted_vals) // 5000)
ax.plot(sorted_vals[::step], cdf[::step], color='#E74C3C', linewidth=1.5)
ax.set_title('Cumulative Distribution', fontsize=12, fontweight='bold')
ax.set_xlabel('Pixel Value')
ax.set_ylabel('Cumulative Proportion')
ax.grid(True, alpha=0.3)

# Panel 3: Top 30 values bar chart
ax = axes[2]
top30 = val_counts_ds.head(30)
ax.barh(range(len(top30)), top30['COUNT'], color='#2980B9', alpha=0.8)
ax.set_yticks(range(len(top30)))
ax.set_yticklabels(top30['VALUE'].astype(str), fontsize=8)
ax.set_xlabel('Pixel Count')
ax.set_title('Top 30 Values', fontsize=12, fontweight='bold')
ax.invert_yaxis()

fig.suptitle('Figure 02: Raw Pixel Value Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'vegetation/02_value_distribution')
plt.show()

In [ ]:
# ── Figure 03: WHRTYPE Classification (requires RAT) ────────────────────

if rat_available:
    # Merge downsampled values with local RAT
    merged = merge_values_with_rat(val_counts_ds, local_rat.reset_index().set_index('VALUE'))
    mapped_frac = merged['WHRTYPE'].notna().sum() / len(merged)
    mapped_pixels = merged.loc[merged['WHRTYPE'].notna(), 'COUNT'].sum()
    total_pixels_ds = merged['COUNT'].sum()
    
    print(f"Mapped values: {merged['WHRTYPE'].notna().sum()} / {len(merged)} ({100*mapped_frac:.1f}%)")
    print(f"Mapped pixels: {mapped_pixels:,} / {total_pixels_ds:,} ({100*mapped_pixels/total_pixels_ds:.1f}%)")
    
    # Aggregate by WHRTYPE
    whrtype_counts = merged.dropna(subset=['WHRTYPE']).groupby('WHRTYPE')['COUNT'].sum()
    whrtype_counts = whrtype_counts.sort_values(ascending=True)
    top30_types = whrtype_counts.tail(30)
    
    # Add names
    labels = [f"{code} — {get_whrtype_name(code)}" for code in top30_types.index]
    
    fig, ax = plt.subplots(figsize=(12, 10))
    colors = [get_lifeform_colors().get(WHRTYPE_TO_LIFEFORM.get(c, ''), '#BDC3C7') for c in top30_types.index]
    ax.barh(range(len(top30_types)), top30_types.values, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(len(top30_types)))
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('Pixel Count (downsampled)', fontsize=11)
    ax.set_title('Figure 03: Top 30 WHRTYPE Classes by Area', fontsize=14, fontweight='bold')
    
    # Legend for lifeform colors
    lf_colors = get_lifeform_colors()
    legend_patches = [mpatches.Patch(color=lf_colors[lf], label=LIFEFORM_MAP.get(lf, lf))
                      for lf in LIFEFORM_ORDER if lf in lf_colors]
    ax.legend(handles=legend_patches, loc='lower right', fontsize=9, title='Lifeform')
    
    plt.tight_layout()
    save_figure(fig, 'vegetation/03_whrtype_classification')
    plt.show()
else:
    print("⚠ Local RAT not available — skipping WHRTYPE classification figure.")
    print("  Run the RAT builder script to generate fveg22_local_rat.csv")

---
## Part 4: Lifeform & WHR13 Classification

In [ ]:
# ── Figure 04: Lifeform Composition ──────────────────────────────────────

if rat_available:
    # Aggregate by LIFEFORM
    lifeform_counts = merged.dropna(subset=['LIFEFORM']).groupby('LIFEFORM')['COUNT'].sum()
    lifeform_counts = lifeform_counts.reindex([lf for lf in LIFEFORM_ORDER if lf in lifeform_counts.index])
    lifeform_total = lifeform_counts.sum()
    lifeform_pct = 100 * lifeform_counts / lifeform_total
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    lf_colors = get_lifeform_colors()
    colors = [lf_colors.get(lf, '#BDC3C7') for lf in lifeform_counts.index]
    labels = [LIFEFORM_MAP.get(lf, lf) for lf in lifeform_counts.index]
    
    # Panel 1: Pie chart
    ax = axes[0]
    wedges, texts, autotexts = ax.pie(
        lifeform_counts.values, labels=None, colors=colors,
        autopct=lambda p: f'{p:.1f}%' if p > 3 else '',
        pctdistance=0.75, startangle=90
    )
    ax.legend(wedges, labels, loc='lower left', fontsize=9)
    ax.set_title('Lifeform Proportions', fontsize=12, fontweight='bold')
    
    # Panel 2: Horizontal bar
    ax = axes[1]
    bars = ax.barh(range(len(lifeform_counts)), lifeform_counts.values, color=colors, edgecolor='white')
    ax.set_yticks(range(len(lifeform_counts)))
    ax.set_yticklabels(labels, fontsize=11)
    ax.set_xlabel('Pixel Count (downsampled)', fontsize=11)
    for i, (count, pct) in enumerate(zip(lifeform_counts.values, lifeform_pct.values)):
        ax.text(count + lifeform_total * 0.005, i, f'{pct:.1f}%', va='center', fontsize=10)
    ax.set_title('Lifeform by Pixel Count', fontsize=12, fontweight='bold')
    
    fig.suptitle('Figure 04: CWHR Lifeform Composition', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_figure(fig, 'vegetation/04_lifeform_composition')
    plt.show()
else:
    print("⚠ Local RAT not available — skipping lifeform figure.")

In [ ]:
# ── Figure 05: WHR13 Classification ──────────────────────────────────────

if rat_available:
    # Aggregate by WHR13NAME
    whr13_counts = merged.dropna(subset=['WHR13NAME']).groupby('WHR13NAME')['COUNT'].sum()
    whr13_counts = whr13_counts.reindex([c for c in WHR13_ORDER if c in whr13_counts.index])
    whr13_total = whr13_counts.sum()
    whr13_pct = 100 * whr13_counts / whr13_total
    
    w13_colors = get_whr13_colors()
    colors = [w13_colors.get(c, '#BDC3C7') for c in whr13_counts.index]
    
    fig, ax = plt.subplots(figsize=(12, 8))
    bars = ax.barh(range(len(whr13_counts)), whr13_counts.values, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(len(whr13_counts)))
    ax.set_yticklabels(whr13_counts.index, fontsize=11)
    ax.set_xlabel('Pixel Count (downsampled)', fontsize=11)
    for i, (count, pct) in enumerate(zip(whr13_counts.values, whr13_pct.values)):
        ax.text(count + whr13_total * 0.005, i, f'{pct:.1f}%', va='center', fontsize=10)
    
    ax.set_title('Figure 05: WHR13 Land Cover Classes', fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_figure(fig, 'vegetation/05_whr13_classification')
    plt.show()
else:
    print("⚠ Local RAT not available — skipping WHR13 figure.")

---
## Part 5: Spatial Analysis

In [ ]:
# ── Build lifeform spatial map ────────────────────────────────────────────

if rat_available:
    # Map each pixel to its lifeform category
    val_to_lifeform = {}
    for val, row in local_rat.iterrows():
        lf = row.get('LIFEFORM')
        if pd.notna(lf):
            val_to_lifeform[val] = lf
    
    # Create lifeform array
    veg_valid = ~np.isnan(veg_ds)
    lifeform_idx = np.full(veg_ds.shape, -1, dtype=np.int8)
    lf_list = LIFEFORM_ORDER
    lf_to_idx = {lf: i for i, lf in enumerate(lf_list)}
    
    for val in np.unique(veg_ds[veg_valid]).astype(int):
        lf = val_to_lifeform.get(val)
        if lf and lf in lf_to_idx:
            mask = (veg_ds == val) & veg_valid
            lifeform_idx[mask] = lf_to_idx[lf]
    
    mapped_pct = 100 * np.sum(lifeform_idx >= 0) / np.sum(veg_valid)
    print(f"Lifeform mapping coverage: {mapped_pct:.1f}% of valid pixels")
else:
    lifeform_idx = None
    print("⚠ Local RAT not available — spatial maps will show raw values.")

In [ ]:
# ── Figure 06: California Vegetation Map ─────────────────────────────────

fig, ax = plt.subplots(figsize=(12, 14))
extent = [meta['bounds'].left, meta['bounds'].right,
          meta['bounds'].bottom, meta['bounds'].top]

if lifeform_idx is not None:
    # Build custom colormap
    lf_colors_map = get_lifeform_colors()
    colors_list = [lf_colors_map.get(lf, '#BDC3C7') for lf in LIFEFORM_ORDER]
    cmap = mcolors.ListedColormap(colors_list)
    norm = mcolors.BoundaryNorm(np.arange(-0.5, len(LIFEFORM_ORDER) + 0.5, 1), len(LIFEFORM_ORDER))
    
    display = np.ma.masked_where(lifeform_idx < 0, lifeform_idx)
    im = ax.imshow(display, cmap=cmap, norm=norm, aspect='equal', extent=extent, interpolation='nearest')
    
    # Legend
    legend_patches = [mpatches.Patch(color=colors_list[i], label=LIFEFORM_MAP.get(lf, lf))
                      for i, lf in enumerate(LIFEFORM_ORDER)]
    ax.legend(handles=legend_patches, loc='lower left', fontsize=9,
              title='Lifeform', framealpha=0.9)
else:
    # Fallback: raw values
    display = np.ma.masked_invalid(veg_ds)
    im = ax.imshow(display, cmap='viridis', aspect='equal', extent=extent, interpolation='nearest')
    plt.colorbar(im, ax=ax, label='Pixel Value', shrink=0.6)

ax.set_title('Figure 06: California Vegetation — CWHR Lifeform Map', fontsize=14, fontweight='bold')
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
plt.tight_layout()
save_figure(fig, 'vegetation/06_california_vegetation_map')
plt.show()

In [ ]:
# ── Figure 07: Regional Vegetation Composition ──────────────────────────

if lifeform_idx is not None:
    h, w = lifeform_idx.shape
    regions = {
        'Northern CA': (0, h // 3),
        'Central CA': (h // 3, 2 * h // 3),
        'Southern CA': (2 * h // 3, h),
    }
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    lf_colors_map = get_lifeform_colors()
    
    for ax, (region_name, (r_start, r_end)) in zip(axes, regions.items()):
        region_data = lifeform_idx[r_start:r_end, :]
        valid = region_data[region_data >= 0]
        
        if len(valid) > 0:
            counts = pd.Series(valid).value_counts().sort_index()
            labels_r = [LIFEFORM_MAP.get(LIFEFORM_ORDER[i], f'Unknown {i}') for i in counts.index]
            colors_r = [lf_colors_map.get(LIFEFORM_ORDER[i], '#BDC3C7') for i in counts.index]
            
            wedges, texts, autotexts = ax.pie(
                counts.values, labels=None, colors=colors_r,
                autopct=lambda p: f'{p:.0f}%' if p > 5 else '',
                pctdistance=0.75, startangle=90, textprops={'fontsize': 9}
            )
            ax.set_title(region_name, fontsize=12, fontweight='bold')
    
    # Shared legend
    legend_patches = [mpatches.Patch(color=lf_colors_map[lf], label=LIFEFORM_MAP.get(lf, lf))
                      for lf in LIFEFORM_ORDER]
    fig.legend(handles=legend_patches, loc='lower center', ncol=4, fontsize=9,
               bbox_to_anchor=(0.5, -0.05))
    
    fig.suptitle('Figure 07: Regional Vegetation Composition (N/Central/S California)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_figure(fig, 'vegetation/07_regional_vegetation_composition')
    plt.show()
else:
    print("⚠ Lifeform mapping not available — skipping regional composition.")

In [ ]:
# ── Figure 08: Spatial Coverage Map ──────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 12))
valid_mask = ~np.isnan(veg_ds)
ax.imshow(valid_mask.astype(float), cmap='Greens', aspect='equal', extent=extent,
          interpolation='nearest', vmin=0, vmax=1)
ax.set_title('Figure 08: FVEG22 Valid Data Spatial Coverage', fontsize=14, fontweight='bold')
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')

valid_pct = 100 * np.sum(valid_mask) / valid_mask.size
ax.annotate(f'Valid: {valid_pct:.1f}%', xy=(0.02, 0.02), xycoords='axes fraction',
            fontsize=11, bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
plt.tight_layout()
save_figure(fig, 'vegetation/08_spatial_coverage_map')
plt.show()

---
## Part 6: Fire-Relevant Vegetation

Each WHRTYPE has been assigned a fire risk rating based on California fire ecology:
- **High**: Chaparral, dry conifer, grassland — California's most fire-prone types
- **Moderate**: Oak woodland, sagebrush, mixed forests — burn under drought conditions  
- **Low**: Moist forests, desert, wetlands — limited fire activity
- **Negligible**: Urban, water, agriculture, barren — non-wildland

In [ ]:
# ── Figure 09: Fire-Prone Vegetation Types ───────────────────────────────

if rat_available:
    # Add fire risk to merged data
    merged['fire_risk'] = merged['WHRTYPE'].map(FIRE_PRONE_WHRTYPE)
    
    # Aggregate by fire risk level
    risk_counts = merged.dropna(subset=['fire_risk']).groupby('fire_risk')['COUNT'].sum()
    risk_counts = risk_counts.reindex([r for r in FIRE_RISK_ORDER if r in risk_counts.index])
    risk_total = risk_counts.sum()
    risk_pct = 100 * risk_counts / risk_total
    
    fr_colors = get_fire_risk_colors()
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # Panel 1: Risk level pie
    ax = axes[0]
    colors = [fr_colors.get(r, '#BDC3C7') for r in risk_counts.index]
    wedges, texts, autotexts = ax.pie(
        risk_counts.values, labels=risk_counts.index, colors=colors,
        autopct='%1.1f%%', pctdistance=0.75, startangle=90,
        textprops={'fontsize': 11}
    )
    ax.set_title('Fire Risk Level Proportions', fontsize=12, fontweight='bold')
    
    # Panel 2: Top fire-prone types
    ax = axes[1]
    high_mod = merged[merged['fire_risk'].isin(['High', 'Moderate'])]
    if len(high_mod) > 0:
        type_risk = high_mod.groupby(['WHRTYPE', 'fire_risk'])['COUNT'].sum().reset_index()
        type_totals = type_risk.groupby('WHRTYPE')['COUNT'].sum().sort_values(ascending=True).tail(20)
        
        bar_colors = []
        for t in type_totals.index:
            risk = FIRE_PRONE_WHRTYPE.get(t, 'Low')
            bar_colors.append(fr_colors.get(risk, '#BDC3C7'))
        
        labels_fr = [f"{c} — {get_whrtype_name(c)}" for c in type_totals.index]
        ax.barh(range(len(type_totals)), type_totals.values, color=bar_colors, edgecolor='white')
        ax.set_yticks(range(len(type_totals)))
        ax.set_yticklabels(labels_fr, fontsize=9)
        ax.set_xlabel('Pixel Count', fontsize=11)
        ax.set_title('Top 20 High/Moderate Risk Types', fontsize=12, fontweight='bold')
    
    fig.suptitle('Figure 09: Fire-Prone Vegetation Classification', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_figure(fig, 'vegetation/09_fire_prone_vegetation')
    plt.show()
else:
    print("⚠ Local RAT not available — skipping fire-prone vegetation figure.")

In [ ]:
# ── Figure 10: Fire-Prone Vegetation Map ─────────────────────────────────

if rat_available:
    # Map pixel values to fire risk index
    val_to_risk = {}
    for val, row in local_rat.iterrows():
        whrtype = row.get('WHRTYPE')
        if pd.notna(whrtype):
            val_to_risk[val] = FIRE_PRONE_WHRTYPE.get(whrtype, 'Unknown')
    
    risk_order = ['High', 'Moderate', 'Low', 'Negligible']
    risk_to_idx = {r: i for i, r in enumerate(risk_order)}
    risk_arr = np.full(veg_ds.shape, -1, dtype=np.int8)
    veg_valid = ~np.isnan(veg_ds)
    
    for val in np.unique(veg_ds[veg_valid]).astype(int):
        risk = val_to_risk.get(val)
        if risk and risk in risk_to_idx:
            mask = (veg_ds == val) & veg_valid
            risk_arr[mask] = risk_to_idx[risk]
    
    fig, ax = plt.subplots(figsize=(12, 14))
    fr_colors = get_fire_risk_colors()
    colors_list = [fr_colors[r] for r in risk_order]
    cmap = mcolors.ListedColormap(colors_list)
    norm = mcolors.BoundaryNorm(np.arange(-0.5, len(risk_order) + 0.5, 1), len(risk_order))
    
    display = np.ma.masked_where(risk_arr < 0, risk_arr)
    ax.imshow(display, cmap=cmap, norm=norm, aspect='equal', extent=extent, interpolation='nearest')
    
    legend_patches = [mpatches.Patch(color=colors_list[i], label=risk_order[i])
                      for i in range(len(risk_order))]
    ax.legend(handles=legend_patches, loc='lower left', fontsize=10,
              title='Fire Risk', framealpha=0.9)
    
    ax.set_title('Figure 10: Fire Risk by Vegetation Type', fontsize=14, fontweight='bold')
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    plt.tight_layout()
    save_figure(fig, 'vegetation/10_fire_prone_map')
    plt.show()
else:
    print("⚠ Local RAT not available — skipping fire-prone map.")

In [ ]:
# ── Figure 11: Fuel Type Classification ──────────────────────────────────

if rat_available:
    # Classify by fuel behavior type
    merged['fuel_type'] = classify_fire_fuel_type(merged['WHRTYPE'].values)
    fuel_counts = merged.dropna(subset=['fuel_type']).groupby('fuel_type')['COUNT'].sum()
    fuel_order = ['Timber', 'Brush/Shrub', 'Grass', 'Non-Fuel', 'Unknown']
    fuel_counts = fuel_counts.reindex([f for f in fuel_order if f in fuel_counts.index])
    fuel_total = fuel_counts.sum()
    
    fuel_colors = {'Timber': '#1B7837', 'Brush/Shrub': '#D9A441',
                   'Grass': '#F7E353', 'Non-Fuel': '#95A5A6', 'Unknown': '#BDC3C7'}
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Panel 1: Bar chart
    ax = axes[0]
    colors = [fuel_colors.get(f, '#BDC3C7') for f in fuel_counts.index]
    bars = ax.bar(range(len(fuel_counts)), fuel_counts.values, color=colors, edgecolor='white')
    ax.set_xticks(range(len(fuel_counts)))
    ax.set_xticklabels(fuel_counts.index, fontsize=11)
    ax.set_ylabel('Pixel Count', fontsize=11)
    for i, (count, ft) in enumerate(zip(fuel_counts.values, fuel_counts.index)):
        pct = 100 * count / fuel_total
        ax.text(i, count + fuel_total * 0.01, f'{pct:.1f}%', ha='center', fontsize=10)
    ax.set_title('Fuel Type Distribution', fontsize=12, fontweight='bold')
    
    # Panel 2: Map
    ax = axes[1]
    val_to_fuel = {}
    for val, row in local_rat.iterrows():
        whrtype = row.get('WHRTYPE')
        if pd.notna(whrtype):
            val_to_fuel[val] = classify_fire_fuel_type(whrtype)
    
    fuel_idx_map = {ft: i for i, ft in enumerate(fuel_order)}
    fuel_arr = np.full(veg_ds.shape, -1, dtype=np.int8)
    veg_valid = ~np.isnan(veg_ds)
    for val in np.unique(veg_ds[veg_valid]).astype(int):
        ft = val_to_fuel.get(val)
        if ft and ft in fuel_idx_map:
            mask = (veg_ds == val) & veg_valid
            fuel_arr[mask] = fuel_idx_map[ft]
    
    fuel_cmap = mcolors.ListedColormap([fuel_colors[f] for f in fuel_order])
    fuel_norm = mcolors.BoundaryNorm(np.arange(-0.5, len(fuel_order) + 0.5, 1), len(fuel_order))
    display = np.ma.masked_where(fuel_arr < 0, fuel_arr)
    ax.imshow(display, cmap=fuel_cmap, norm=fuel_norm, aspect='equal', extent=extent, interpolation='nearest')
    fuel_patches = [mpatches.Patch(color=fuel_colors[f], label=f) for f in fuel_order if f in fuel_counts.index]
    ax.legend(handles=fuel_patches, loc='lower left', fontsize=9, framealpha=0.9)
    ax.set_title('Fuel Type Map', fontsize=12, fontweight='bold')
    
    fig.suptitle('Figure 11: Fire Fuel Type Classification', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_figure(fig, 'vegetation/11_fuel_type_classification')
    plt.show()
else:
    print("⚠ Local RAT not available — skipping fuel type figure.")

---
## Part 7: Vegetation Structure

CWHR encodes structural information in WHRSIZE (1–6, X) and WHRDENSITY (S, P, M, D, X):
- **Size classes** reflect tree diameter at breast height (DBH), indicating structural maturity
- **Density classes** reflect canopy closure percentage
- Non-tree types (shrub, grassland, etc.) use "X" for both fields

In [ ]:
# ── Figure 12: Size Structure Analysis ────────────────────────────────────

if rat_available:
    # Filter to tree types (size != X)
    tree_data = merged[merged['WHRSIZE'].isin(['1', '2', '3', '4', '5', '6'])]
    
    if len(tree_data) > 0:
        size_counts = tree_data.groupby('WHRSIZE')['COUNT'].sum()
        size_counts = size_counts.reindex([s for s in WHRSIZE_ORDER if s in size_counts.index and s != 'X'])
        size_total = size_counts.sum()
        
        size_colors = ['#BFEE9F', '#82E0AA', '#2ECC71', '#27AE60', '#1B7837', '#145A2C']
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # Panel 1: Bar chart
        ax = axes[0]
        labels = [WHRSIZE_MAP.get(s, s) for s in size_counts.index]
        ax.bar(range(len(size_counts)), size_counts.values,
               color=size_colors[:len(size_counts)], edgecolor='white')
        ax.set_xticks(range(len(size_counts)))
        ax.set_xticklabels(labels, fontsize=9, rotation=30, ha='right')
        ax.set_ylabel('Pixel Count', fontsize=11)
        for i, count in enumerate(size_counts.values):
            pct = 100 * count / size_total
            ax.text(i, count + size_total * 0.01, f'{pct:.1f}%', ha='center', fontsize=10)
        ax.set_title('Tree Size Class Distribution', fontsize=12, fontweight='bold')
        
        # Panel 2: Size by lifeform (conifer vs hardwood)
        ax = axes[1]
        tree_by_lf = tree_data.groupby(['WHRSIZE', 'LIFEFORM'])['COUNT'].sum().unstack(fill_value=0)
        lf_in_tree = [lf for lf in ['CONIFER', 'HARDWOOD'] if lf in tree_by_lf.columns]
        if lf_in_tree:
            tree_by_lf = tree_by_lf[lf_in_tree]
            tree_by_lf = tree_by_lf.reindex([s for s in WHRSIZE_ORDER if s in tree_by_lf.index and s != 'X'])
            tree_by_lf.plot(kind='bar', ax=ax, color=[get_lifeform_colors()[lf] for lf in lf_in_tree],
                           edgecolor='white')
            ax.set_xticklabels([WHRSIZE_MAP.get(s, s)[:20] for s in tree_by_lf.index],
                              fontsize=9, rotation=30, ha='right')
            ax.set_ylabel('Pixel Count', fontsize=11)
            ax.legend(title='Lifeform', fontsize=9)
            ax.set_title('Size by Lifeform', fontsize=12, fontweight='bold')
        
        fig.suptitle('Figure 12: CWHR Tree Size Structure', fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        save_figure(fig, 'vegetation/12_size_structure_analysis')
        plt.show()
    else:
        print("No tree-type data found with size classes.")
else:
    print("⚠ Local RAT not available — skipping size structure figure.")

In [ ]:
# ── Figure 13: Density Structure Analysis ────────────────────────────────

if rat_available:
    tree_data = merged[merged['WHRDENSITY'].isin(['S', 'P', 'M', 'D'])]
    
    if len(tree_data) > 0:
        dens_counts = tree_data.groupby('WHRDENSITY')['COUNT'].sum()
        dens_counts = dens_counts.reindex([d for d in WHRDENSITY_ORDER if d in dens_counts.index and d != 'X'])
        dens_total = dens_counts.sum()
        
        dens_colors = ['#EAFFBF', '#8CC96D', '#3B992B', '#0F5C0B']
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # Panel 1: Bar chart
        ax = axes[0]
        labels = [WHRDENSITY_MAP.get(d, d) for d in dens_counts.index]
        ax.bar(range(len(dens_counts)), dens_counts.values,
               color=dens_colors[:len(dens_counts)], edgecolor='white')
        ax.set_xticks(range(len(dens_counts)))
        ax.set_xticklabels(labels, fontsize=10)
        ax.set_ylabel('Pixel Count', fontsize=11)
        for i, count in enumerate(dens_counts.values):
            pct = 100 * count / dens_total
            ax.text(i, count + dens_total * 0.01, f'{pct:.1f}%', ha='center', fontsize=10)
        ax.set_title('Canopy Density Distribution', fontsize=12, fontweight='bold')
        
        # Panel 2: Density by WHR13
        ax = axes[1]
        dens_by_whr13 = tree_data.groupby(['WHRDENSITY', 'WHR13NAME'])['COUNT'].sum().unstack(fill_value=0)
        top_whr13 = dens_by_whr13.sum(axis=0).nlargest(5).index
        dens_by_whr13 = dens_by_whr13[top_whr13]
        dens_by_whr13 = dens_by_whr13.reindex([d for d in WHRDENSITY_ORDER if d in dens_by_whr13.index and d != 'X'])
        w13_colors = get_whr13_colors()
        dens_by_whr13.plot(kind='bar', ax=ax,
                           color=[w13_colors.get(c, '#BDC3C7') for c in top_whr13],
                           edgecolor='white')
        ax.set_xticklabels([WHRDENSITY_MAP.get(d, d)[:20] for d in dens_by_whr13.index],
                          fontsize=10, rotation=0)
        ax.set_ylabel('Pixel Count', fontsize=11)
        ax.legend(title='WHR13', fontsize=8, loc='upper right')
        ax.set_title('Density by Top 5 WHR13 Classes', fontsize=12, fontweight='bold')
        
        fig.suptitle('Figure 13: CWHR Canopy Density Structure', fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        save_figure(fig, 'vegetation/13_density_structure_analysis')
        plt.show()
    else:
        print("No tree-type data found with density classes.")
else:
    print("⚠ Local RAT not available — skipping density structure figure.")

---
## Part 8: Fire Overlay Analysis

Overlaying fire occurrence data (CAL FIRE perimeters, MTBS centroids) with vegetation
reveals which vegetation types burn most frequently and most severely.

In [ ]:
# ── Load fire data ───────────────────────────────────────────────────────

# CAL FIRE perimeters (1993+)
from src.utils.data_loader import load_fire_perimeters, add_derived_columns
try:
    fires = load_fire_perimeters()
    fires = add_derived_columns(fires)
    fires_hq = fires[fires['YEAR_'] >= HIGH_QUALITY_CUTOFF].copy()
    fires_hq = fires_hq.to_crs(CRS_ALBERS)
    print(f"CAL FIRE perimeters (1993+): {len(fires_hq):,}")
except Exception as e:
    fires_hq = None
    print(f"Could not load CAL FIRE data: {e}")

# MTBS centroids
from src.utils.mtbs_data_loader import load_mtbs_fod, add_mtbs_derived_columns
try:
    mtbs = load_mtbs_fod(state='CA')
    mtbs = add_mtbs_derived_columns(mtbs)
    mtbs_albers = mtbs.to_crs(CRS_ALBERS)
    print(f"MTBS centroids (CA): {len(mtbs_albers):,}")
except Exception as e:
    mtbs_albers = None
    print(f"Could not load MTBS data: {e}")

In [ ]:
# ── Create burned mask from CAL FIRE perimeters ──────────────────────────

if fires_hq is not None:
    print("Creating burned area mask (downsampled resolution)...")
    h, w = veg_ds.shape
    burned_mask = np.zeros((h, w), dtype=bool)
    
    # Use geometry_mask with downsampled transform
    try:
        geoms = fires_hq.geometry.values
        burned_mask = ~geometry_mask(geoms, out_shape=(h, w), transform=veg_transform, invert=False)
        burned_valid = burned_mask & (~np.isnan(veg_ds))
        unburned_valid = (~burned_mask) & (~np.isnan(veg_ds))
        
        print(f"Burned pixels: {np.sum(burned_valid):,}")
        print(f"Unburned pixels: {np.sum(unburned_valid):,}")
        print(f"Burned fraction of valid: {100 * np.sum(burned_valid) / np.sum(~np.isnan(veg_ds)):.1f}%")
    except Exception as e:
        print(f"Error creating burned mask: {e}")
        burned_mask = None
else:
    burned_mask = None
    print("No fire data available for overlay analysis.")

In [ ]:
# ── Figure 14: Burned vs Unburned Vegetation Composition ─────────────────

if rat_available and burned_mask is not None and lifeform_idx is not None:
    burned_valid = burned_mask & (~np.isnan(veg_ds))
    unburned_valid = (~burned_mask) & (~np.isnan(veg_ds))
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    lf_colors_map = get_lifeform_colors()
    
    for ax, (label, mask) in zip(axes, [('Burned Areas', burned_valid), ('Unburned Areas', unburned_valid)]):
        lf_data = lifeform_idx[mask]
        lf_data = lf_data[lf_data >= 0]
        if len(lf_data) > 0:
            counts = pd.Series(lf_data).value_counts().sort_index()
            labels_lf = [LIFEFORM_MAP.get(LIFEFORM_ORDER[i], f'{i}') for i in counts.index]
            colors_lf = [lf_colors_map.get(LIFEFORM_ORDER[i], '#BDC3C7') for i in counts.index]
            ax.pie(counts.values, labels=None, colors=colors_lf,
                   autopct=lambda p: f'{p:.1f}%' if p > 3 else '',
                   pctdistance=0.75, startangle=90, textprops={'fontsize': 9})
            ax.set_title(f'{label}\n({len(lf_data):,} pixels)', fontsize=12, fontweight='bold')
    
    legend_patches = [mpatches.Patch(color=lf_colors_map[lf], label=LIFEFORM_MAP.get(lf, lf))
                      for lf in LIFEFORM_ORDER]
    fig.legend(handles=legend_patches, loc='lower center', ncol=4, fontsize=9,
               bbox_to_anchor=(0.5, -0.05))
    
    fig.suptitle('Figure 14: Vegetation Composition — Burned vs Unburned',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_figure(fig, 'vegetation/14_vegetation_burned_areas')
    plt.show()
else:
    print("⚠ Skipping burned/unburned analysis (requires RAT + fire data).")

In [ ]:
# ── Figure 15: Vegetation-Fire Crosstab ──────────────────────────────────

if rat_available and burned_mask is not None:
    # Build crosstab: WHR13 × burned/unburned
    burned_valid = burned_mask & (~np.isnan(veg_ds))
    unburned_valid = (~burned_mask) & (~np.isnan(veg_ds))
    
    # Map values to WHR13
    val_to_whr13 = {}
    for val, row in local_rat.iterrows():
        whr13 = row.get('WHR13NAME')
        if pd.notna(whr13):
            val_to_whr13[val] = whr13
    
    burned_whr13 = []
    unburned_whr13 = []
    
    for val in np.unique(veg_ds[~np.isnan(veg_ds)]).astype(int):
        whr13 = val_to_whr13.get(val)
        if whr13:
            b_count = np.sum((veg_ds == val) & burned_valid)
            u_count = np.sum((veg_ds == val) & unburned_valid)
            if b_count > 0:
                burned_whr13.append({'WHR13': whr13, 'count': b_count})
            if u_count > 0:
                unburned_whr13.append({'WHR13': whr13, 'count': u_count})
    
    burned_df = pd.DataFrame(burned_whr13).groupby('WHR13')['count'].sum()
    unburned_df = pd.DataFrame(unburned_whr13).groupby('WHR13')['count'].sum()
    
    # Compute burn ratio
    crosstab = pd.DataFrame({
        'Burned': burned_df,
        'Unburned': unburned_df,
    }).fillna(0)
    crosstab['Total'] = crosstab['Burned'] + crosstab['Unburned']
    crosstab['Burn_Rate_%'] = 100 * crosstab['Burned'] / crosstab['Total']
    crosstab = crosstab.reindex([c for c in WHR13_ORDER if c in crosstab.index])
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # Panel 1: Burn rate by WHR13
    ax = axes[0]
    w13_colors = get_whr13_colors()
    colors = [w13_colors.get(c, '#BDC3C7') for c in crosstab.index]
    ax.barh(range(len(crosstab)), crosstab['Burn_Rate_%'], color=colors, edgecolor='white')
    ax.set_yticks(range(len(crosstab)))
    ax.set_yticklabels(crosstab.index, fontsize=10)
    ax.set_xlabel('Burn Rate (%)', fontsize=11)
    ax.set_title('Burn Rate by WHR13 Class', fontsize=12, fontweight='bold')
    
    # Panel 2: Composition comparison (stacked)
    ax = axes[1]
    burned_pct = 100 * crosstab['Burned'] / crosstab['Burned'].sum()
    unburned_pct = 100 * crosstab['Unburned'] / crosstab['Unburned'].sum()
    
    x = np.arange(len(crosstab))
    width = 0.35
    ax.barh(x - width/2, burned_pct, width, color='#E74C3C', alpha=0.8, label='Burned')
    ax.barh(x + width/2, unburned_pct, width, color='#3498DB', alpha=0.8, label='Unburned')
    ax.set_yticks(x)
    ax.set_yticklabels(crosstab.index, fontsize=10)
    ax.set_xlabel('% of Category', fontsize=11)
    ax.legend(fontsize=10)
    ax.set_title('Burned vs Unburned Composition', fontsize=12, fontweight='bold')
    
    fig.suptitle('Figure 15: Vegetation Type × Fire Occurrence', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_figure(fig, 'vegetation/15_vegetation_fire_crosstab')
    plt.show()
    
    # Print table
    print("\nBurn Rate by WHR13 Class:")
    print(crosstab[['Burned', 'Unburned', 'Burn_Rate_%']].to_string())
else:
    print("⚠ Skipping fire crosstab (requires RAT + fire data).")

---
## Part 9: 800m Grid & ML Readiness

### Vegetation Features for ML Model

At the target 800m × 800m grid resolution, FVEG22 provides:

| Feature | Description | Computation |
|---------|-------------|-------------|
| `dominant_whrtype` | Most common WHRTYPE in cell | Mode of pixel values |
| `dominant_lifeform` | Most common lifeform category | Mode after lifeform mapping |
| `dominant_whr13` | Most common WHR13 class | Mode after WHR13 mapping |
| `fire_risk_dominant` | Fire risk of dominant type | Lookup from FIRE_PRONE_WHRTYPE |
| `timber_fraction` | Proportion of timber fuel | Count of timber pixels / total |
| `brush_fraction` | Proportion of brush fuel | Count of brush pixels / total |
| `grass_fraction` | Proportion of grass fuel | Count of grass pixels / total |
| `veg_diversity` | Number of unique WHRTYPEs in cell | Count of unique types |
| `mean_tree_size` | Average tree size class (1–6) | Mean of WHRSIZE where applicable |
| `mean_density` | Average canopy density class | Ordinal mean of WHRDENSITY |

### Integration with Other Datasets

Vegetation × Topography interactions are strong fire predictors:
- **Chaparral on steep south-facing slopes** → extreme fire risk
- **Dense conifer at high elevation** → crown fire potential under drought
- **Grassland in valleys** → fast-spreading surface fire

In [ ]:
# ── Figure 16: 800m Grid Preview ──────────────────────────────────────────

# Compute a coarser preview (~800m from 30m ≈ 27x factor)
print("Computing 800m preview...")
preview_ds, preview_transform, preview_meta = load_vegetation_downsampled(factor=27)
print(f"Preview shape: {preview_meta['downsampled_shape']} ({preview_meta['effective_resolution_m']:.0f}m)")

fig = plt.figure(figsize=(16, 12))
gs = GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.25)

preview_extent = [meta['bounds'].left, meta['bounds'].right,
                  meta['bounds'].bottom, meta['bounds'].top]

# Panel 1: 40x visualization resolution
ax1 = fig.add_subplot(gs[0, 0])
display_40x = np.ma.masked_invalid(veg_ds)
ax1.imshow(display_40x, cmap='viridis', aspect='equal', extent=preview_extent, interpolation='nearest')
ax1.set_title(f'40x Downsample ({veg_meta["effective_resolution_m"]:.0f}m)', fontsize=11, fontweight='bold')
ax1.set_xlabel('Easting (m)')

# Panel 2: ~800m (27x) preview
ax2 = fig.add_subplot(gs[0, 1])
display_27x = np.ma.masked_invalid(preview_ds)
ax2.imshow(display_27x, cmap='viridis', aspect='equal', extent=preview_extent, interpolation='nearest')
ax2.set_title(f'27x Downsample ({preview_meta["effective_resolution_m"]:.0f}m ≈ 800m)',
              fontsize=11, fontweight='bold')
ax2.set_xlabel('Easting (m)')

# Panel 3: Resolution comparison text
ax3 = fig.add_subplot(gs[1, 0])
ax3.axis('off')
res_data = [
    ['Native', '30m', f'{meta["height"]:,} × {meta["width"]:,}', '2.81 GB'],
    ['Visualization', f'{veg_meta["effective_resolution_m"]:.0f}m',
     f'{veg_meta["downsampled_shape"][0]:,} × {veg_meta["downsampled_shape"][1]:,}', '~1.7 MB'],
    ['ML Grid Target', '800m',
     f'{preview_meta["downsampled_shape"][0]:,} × {preview_meta["downsampled_shape"][1]:,}', '~3.9 MB'],
]
table = ax3.table(cellText=res_data,
                  colLabels=['Resolution', 'Pixel Size', 'Dimensions', 'Memory'],
                  loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2980B9')
        cell.set_text_props(color='white', fontweight='bold')
ax3.set_title('Resolution Comparison', fontsize=11, fontweight='bold')

# Panel 4: ML feature list
ax4 = fig.add_subplot(gs[1, 1])
ax4.axis('off')
features = [
    ['dominant_whrtype', 'Categorical', 'Most common vegetation type'],
    ['dominant_lifeform', 'Categorical', 'Top lifeform category'],
    ['fire_risk', 'Ordinal', 'High/Moderate/Low/Negligible'],
    ['timber_fraction', 'Continuous', 'Proportion timber fuel'],
    ['brush_fraction', 'Continuous', 'Proportion brush fuel'],
    ['grass_fraction', 'Continuous', 'Proportion grass fuel'],
    ['veg_diversity', 'Count', 'Unique WHRTYPEs per cell'],
]
table2 = ax4.table(cellText=features,
                   colLabels=['Feature', 'Type', 'Description'],
                   loc='center', cellLoc='left')
table2.auto_set_font_size(False)
table2.set_fontsize(10)
table2.scale(1.2, 1.6)
for (row, col), cell in table2.get_celld().items():
    if row == 0:
        cell.set_facecolor('#27AE60')
        cell.set_text_props(color='white', fontweight='bold')
ax4.set_title('ML Features from FVEG22', fontsize=11, fontweight='bold')

fig.suptitle('Figure 16: 800m Grid Preview & ML Feature Summary', fontsize=14, fontweight='bold', y=0.98)
save_figure(fig, 'vegetation/16_800m_grid_preview')
plt.show()

---
## Summary & Next Steps

### Key Findings
1. **Vegetation coverage**: FVEG22 covers ~30.6% of the raster extent (California land area)
2. **65 WHRTYPE classes** collapsed into 13 WHR13 classes and 8 lifeform categories
3. **Fire-prone vegetation**: Chaparral, dry conifer, and grassland types dominate fire-affected areas
4. **Structural detail**: Tree size class and canopy density available for forested types
5. **Burn rate variation**: Significant differences in fire frequency across vegetation types

### ML Integration (Phase 3)
- FVEG22 provides **what burns** — pair with topography (**where it burns**) and climate (**when it burns**)
- Fuel type classification (Timber/Brush/Grass/Non-Fuel) directly maps to fire behavior models
- Vegetation diversity and structure metrics enhance grid-cell characterization
- Cross-reference with LANDFIRE CC for canopy density × vegetation type interactions